# 12a — Lay of the land

The third interlude (convention: 020 §The book): only what CHANGED since
11a. Step 11 widened the base pack with statement `if`/`for` and the array
indexing surface — the two biggest holes in ch07a's original refusal list.

In [1]:
import numpy as np

import pdum.dsl  # noqa: F401
from pdum.dsl.kernel import types as T
from pdum.dsl.kernel.api import jit
from pdum.dsl.kernel.lower import lower_handle
from pdum.dsl.kernel.ops import CORE_OPS
from pdum.dsl.kernel.registry import DEFAULT

rules = dict(DEFAULT.lower_rules)
print("the pack now handles", len([k for k in rules if isinstance(k, type)]), "syntax forms:")
for node_type in rules:
    if isinstance(node_type, type):
        print("  ✅ ast." + node_type.__name__)

the pack now handles 21 syntax forms:
  ✅ ast.Expr
  ✅ ast.Assign
  ✅ ast.AugAssign
  ✅ ast.Return
  ✅ ast.If
  ✅ ast.For
  ✅ ast.While
  ✅ ast.Break
  ✅ ast.Continue
  ✅ ast.Pass
  ✅ ast.Constant
  ✅ ast.Name
  ✅ ast.BinOp
  ✅ ast.UnaryOp
  ✅ ast.BoolOp
  ✅ ast.Compare
  ✅ ast.IfExp
  ✅ ast.Tuple
  ✅ ast.Subscript
  ✅ ast.Attribute
  ✅ ast.Call


In [2]:
# ch07a's remaining refusals, re-run against today's pack:
@jit()
def wants_if_stmt(x):
    y = 0.0
    if x > 0.0:
        y = 1.0 / x
    else:
        y = 0.0
    return y


@jit()
def wants_for(x):
    total = x
    for i in range(4):
        total = total + x
    return total


table = np.ones((2, 2))


@jit()
def wants_array_index(i, j):
    return table[i, j]


@jit()
def wants_while(x):
    while x > 0.0:
        x = x - 1.0
    return x


@jit()
def wants_early_return(x):
    if x > 0.0:
        return x
    return -x


ops = {**CORE_OPS, **DEFAULT.ops}  # the registry rides the CONTEXT seam now (130 §7)
for h, sig in (
    (wants_if_stmt, (T.f64,)),
    (wants_for, (T.f64,)),
    (wants_array_index, (T.i64, T.i64)),
    (wants_while, (T.f64,)),
    (wants_early_return, (T.f64,)),
):
    try:
        lower_handle(h, rules, ops, arg_types=sig, context={"registry": DEFAULT})
        print(f"  ✅ {h.fntype.template.label}: lowers now (step 11)")
    except Exception as e:
        print(f"  🚫 {type(e).__name__}: {str(e)[:95]}")

  ✅ wants_if_stmt: lowers now (step 11)
  ✅ wants_for: lowers now (step 11)
  🚫 NameFateError: 'table' is neither a parameter, a local, nor a capture [/var/folders/vw/tdwty7n902d81s2lj6nz2y9
  🚫 MissingRule: `while` is not in the base pack: bounded `for i in range(...)` only — every serious kernel lang
  🚫 MissingRule: single tail return: `return` must be the function body's last statement, not inside a branch or


## The delta table

| construct | 11a | now |
|---|---|---|
| `if`/`else` statements | 🏗️ | ✅ `core.if` joins; STRICT (same type both paths, no fixpoint) |
| `for i in range(...)` | 🏗️ | ✅ `core.for`, loop carries; zero-trip defined; loop var dies |
| `a[i, j]` on arrays | 🏗️ | ✅ captures only; strict i64; every axis exactly once |
| `a.isel(y=i, x=j)` | — | ✅ NAMED axes (xarray exercise); mandatory on named arrays |
| early `return` | 🏗️ | 🚫 **policy**: single tail return (Taichi's side of the line) |
| `while` / `break` / `continue` | 🏗️ | 🚫 **policy**: bounded loops only (R11's line) |

Note the status change: 11a's 🏗️ ("machinery pending") became either ✅ or
🚫-by-policy. Nothing is left pending *by accident* in the statement
surface; what remains out is out on purpose, with the reason in the error
message. Still out as recorded CUTS (100 §6): array arguments and results,
views/slices/partial `isel`, `sel` (labels are host-side values),
comprehensions (the step-8 discussion: sugar over `core.for`, revisit with
transforms).